# Building a Robust Preprocessing Pipeline with Missing Value Imputation, Outlier Detection, Memory Optimization, and Categorical Encoding
## Introduction:
Data preprocessing is a critical step in the data science pipeline, ensuring that raw data is clean, consistent, and ready for modeling. In this task, we focus on building a robust and comprehensive preprocessing workflow using the Titanic dataset, which provides a realistic mix of numerical and categorical features along with common data quality challenges. The dataset contains missing values, potential outliers, mixed data types, and categorical variables—making it ideal for hands-on experience with data cleaning and preparation techniques.

The goal is to apply multiple preprocessing strategies, including:

- Handling missing values using mean, median, and KNN imputation,
- Identifying and treating outliers using statistical methods like Z-score and IQR,
- Optimizing memory usage by downcasting data types,
- Encoding categorical variables using One-Hot Encoding and Target Encoding methods.

This end-to-end preprocessing will enable better performance and accuracy in downstream machine learning models.



In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
# Load Titanic dataset
df = sns.load_dataset('titanic')

In [3]:
# Display shape and first few rows
print(df.shape)

(891, 15)


In [4]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [5]:
# Missing values
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [6]:
df['age'].fillna(df['age'].mean(), inplace=True)

C:\Users\hg757\AppData\Local\Temp\ipykernel_46408\1503503937.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['age'].fillna(df['age'].mean(), inplace=True)


In [7]:
df['fare'].fillna(df['fare'].median(), inplace=True)

C:\Users\hg757\AppData\Local\Temp\ipykernel_46408\4242156193.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['fare'].fillna(df['fare'].median(), inplace=True)


In [8]:
from sklearn.impute import KNNImputer

In [9]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
knn_imputer = KNNImputer(n_neighbors=3)
df[numeric_cols] = knn_imputer.fit_transform(df[numeric_cols])

In [10]:
from scipy.stats import zscore

In [11]:
z_scores = zscore(df[numeric_cols])
abs_z_scores = np.abs(z_scores)
df = df[(abs_z_scores < 3).all(axis=1)]  # remove rows with Z > 3

In [12]:
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

df = df[~((df[numeric_cols] < (Q1 - 1.5 * IQR)) | 
          (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]


In [13]:
def optimize_memory(df):
    for col in df.select_dtypes(include=['float']):
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int']):
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

df = optimize_memory(df)
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 558 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     558 non-null    float32 
 1   pclass       558 non-null    float32 
 2   sex          558 non-null    object  
 3   age          558 non-null    float32 
 4   sibsp        558 non-null    float32 
 5   parch        558 non-null    float32 
 6   fare         558 non-null    float32 
 7   embarked     558 non-null    object  
 8   class        558 non-null    category
 9   who          558 non-null    object  
 10  adult_male   558 non-null    bool    
 11  deck         69 non-null     category
 12  embark_town  558 non-null    object  
 13  alive        558 non-null    object  
 14  alone        558 non-null    bool    
dtypes: bool(2), category(2), float32(6), object(5)
memory usage: 41.9+ KB


In [14]:
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

In [30]:
# Mean encoding / Target encoding
target_mean = df.groupby('class')['survived'].mean()
df['class_encoded'] = df['class'].map(target_mean)


C:\Users\hg757\AppData\Local\Temp\ipykernel_46408\1253583580.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_mean = df.groupby('class')['survived'].mean()


In [32]:
# Drop original class
df.drop('class', axis=1, inplace=True)

In [36]:
df

,survived,pclass,age,sibsp,parch,fare,who,adult_male,deck,embark_town,alive,alone,sex_male,embarked_Q,embarked_S,class_encoded
0,0.0,3.0,22.000000,1.0,0.0,7.250000,man,True,NaN,Southampton,no,False,True,False,True,0.215847
2,1.0,3.0,26.000000,0.0,0.0,7.925000,woman,False,NaN,Southampton,yes,True,False,False,True,0.215847
3,1.0,1.0,35.000000,1.0,0.0,53.099998,woman,False,C,Southampton,yes,False,False,False,True,0.513514
4,0.0,3.0,35.000000,0.0,0.0,8.050000,man,True,NaN,Southampton,no,True,True,False,True,0.215847
5,0.0,3.0,29.699118,0.0,0.0,8.458300,man,True,NaN,Queenstown,no,True,True,True,False,0.215847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884,0.0,3.0,25.000000,0.0,0.0,7.050000,man,True,NaN,Southampton,no,True,True,False,True,0.215847
886,0.0,2.0,27.000000,0.0,0.0,13.000000,man,True,NaN,Southampton,no,True,True,False,True,0.381356
887,1.0,1.0,19.000000,0.0,0.0,30.000000,woman,False,B,Southampton,yes,True,False,False,True,0.513514
889,1.0,1.0,26.000000,0.0,0.0,30.000000,man,True,C,Cherbourg,yes,True,True,False,False,0.513514


## Conclusion:
This task provided a structured approach to building a robust data preprocessing pipeline. By systematically handling missing values, outliers, memory usage, and categorical variables, we have prepared a clean and efficient dataset suitable for advanced machine learning tasks. The use of both classical (mean/median) and modern (KNN imputation, target encoding) techniques ensures flexibility and scalability for real-world datasets. The cleaned Titanic dataset now stands as a strong foundation for building predictive models with higher accuracy and reliability.